## 모델 저장과 활용

지금까지 만든 모델을 어떻게 남기고, 다시 사용하는 지 다루어보기\
코드 자체는 2줄

```python
import joblib

# 저장
joblib.dump(모델, "파일명")

# 불러오기
model = joblib.load("파일명")
```

In [1]:
import os
import time

import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

from _style import setup
from _ml import build_dataset, time_split, FEATURES, SEED

setup()
pd.set_option("display.width", 130)

df = build_dataset()
train, test, _ = time_split(df)

tr = train.sample(8000, random_state=SEED)
te = test.sample(2000, random_state=SEED)

X_train, y_train = tr[FEATURES], tr["target_up"]
X_test, y_test = te[FEATURES], te["target_up"]

print(f" 학습 {len(tr):,} 행 / 테스트 {len(te):,} 행 / 피처 {len(FEATURES)} 개")

 학습 8,000 행 / 테스트 2,000 행 / 피처 7 개


|| 학습 | 예측 |
|---|---|---|
|소요시간| 몇 분 ~ 몇 시간 | **밀리초(ms)** |
|빈도| 주 1회, 월 1회 | **초당 수백번** |
|필요한 것| 전체 학습 데이터 | 모델 + 입력 하나 |

In [2]:
# 전처리 + 모델 (pipeline)
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandomForestClassifier(n_estimators=100, max_depth=8, random_state=SEED, n_jobs=-1))
])

# 학습 시간 측정
t0 = time.perf_counter()
pipe.fit(X_train, y_train)
train_s = time.perf_counter() - t0

# 예측 시간 측정 (1건)
single_row = X_test.head(1)

pipe.predict(single_row)    # 워밍업: 초기화 비용 측정에서 제외

t1 = time.perf_counter()
for _ in range(50):
    pipe.predict(single_row)
predict_s = (time.perf_counter() - t1) / 50     # 여러변(50회) 반복한 후 평균 계산

print(f"학습 1회 : {train_s * 1000:.1f} ms")
print(f"예측 1건 : {predict_s * 1000:.1f} ms")
print(f"차이    {train_s / predict_s:.0f} 배")
print()
print(f"정확도   {pipe.score(X_test, y_test):.4f}")

학습 1회 : 152.4 ms
예측 1건 : 15.5 ms
차이    10 배

정확도   0.5005


> 💡 **요청마다 학습하면 안 되는 이유**\
> 학습은 예측 1건보다 훨씬 오래 걸리고, 데이터가 커질수록 더 길어질 것임\
> **학습은 미리 해두고, 결과만 꺼내어 사용하는 것**이 저장하는 가장 큰 목적(이유) 임

---

## joblib 으로 저장하고 불러오기

```python
joblib.dump(model, "model.plk")     # 저장
model = joblib.load("model.pkl")    # 불러오기
```

두 줄이 전부임

In [7]:
# model 파일을 저장하기 위한 폴더 생성 (models)
os.makedirs("models", exist_ok=True)

# --- 저장 --------------------------------------------------------------------------------------
joblib.dump(pipe, "models/pipeline.pkl")

size_kb = os.path.getsize("models/pipeline.pkl") / 1024
print(f"저장 완료 : models/pipeline.pkl({size_kb:,.0f} KB)")

# --- 불러오기 ----------------------------------------------------------------------------------
loaded = joblib.load("models/pipeline.pkl")

# 원래 모델과 같은 예측을 하는지 확인
org_pred = pipe.predict(X_test)
loaded_pred = loaded.predict(X_test)

print("불러오기 완료")
print(f"예측 결과 비교 : {np.array_equal(org_pred, loaded_pred)}")

저장 완료 : models/pipeline.pkl(1,506 KB)
불러오기 완료
예측 결과 비교 : True


> ⚠️ **모델은 서버가 구동될 때 한번만 로드**\
> 파일을 읽는 것도 비용이기 때문에, 요청마다 `joblib.load`를 하지말고,\
> 시작할 때 한 번 로드하여 메모리에 들고 있어야 함


> ⚠️ **출처를 모르는 pkl 파일은 열지 말자!**\
> `pickle` 형식은 **불러올 때 코드가 실행될 수 있는 구조** 이기 때문에\
> 내부적으로 만든 파일, 신뢰할 수 있는 출처의 파일만 사용하자.

---

## 스케일러 유/무 비교
`학습할 때      원본 데이터 + StandardScaler    --> 모델`
`예측 할 때     새 데이터 ....                  --> 모델`

**스케일러가 없으면 새 데이터를 어떤 기준으로 변환할지 알 수 없음**



In [11]:
# [1] 스케일러와 모델을 따로 두고, 모델만 저장
scaler = StandardScaler().fit(X_train)

model = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=SEED, n_jobs=-1)

# 학습
model.fit(scaler.transform(X_train), y_train)

# * 스케일링하여 예측
pred_scaled = model.predict(scaler.transform(X_test))

# * 스케일링 없이 예측
pred_unscaled = model.predict(X_test.values)

print(f"{'':<20}{'정확도':>13}")
print("-" * 60)
print(f"{'스케일링 후 예측':<19} {accuracy_score(y_test, pred_scaled):>10.4f}")
print(f"{'스케일링 없이 예측':<18} {accuracy_score(y_test, pred_unscaled):>10.4f}")
print()
print(f"예측이 달라진 건수: {(pred_scaled != pred_unscaled).sum():,} / {len(pred_scaled):,}"
      f"({(pred_scaled != pred_unscaled).mean():.1%})")

                              정확도
------------------------------------------------------------
스케일링 후 예측               0.5005
스케일링 없이 예측             0.5040

예측이 달라진 건수: 877 / 2,000(43.9%)


> ⚠️ **오류가 발생하지 않음!**
> 
> 예측 결과가 **43% 달라졌지만** 아무런 경고도 없음\
> 정확도만 보면 오히려 조금 높게 확인 됨 (우연)\
>
> 동작은 하지만 결과가 틀린 유형의 문제라고 볼 수 있음 (=> 데이터 누수)

---

## Pipeline 을 통해 통째로 저장
```python
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", RandeomForestClassifier())
])

joblib.dump(pipe, "파일명")
```

**전처리와 모델이 한 덩어리로 저장 됨**\
파일을 통해 모델을 불러오면 `predict` 한 번에 스케일링까지 자동으로 처리 됨

---

## 피처 순서

모델의 경우 열 이름을 알 수 없음\
**위치로만** 판단 함

거래량과 등락률의 순서를 바꾸면 어떻게 될지?

In [17]:
import warnings

# 열 순서를 거꾸로 변경
reversed_X = X_test[FEATURES[::-1]]

print(f"원래 순서 : {FEATURES[:3]} ...")
print(f"뒤집은 순서 : {FEATURES[::-1][:3]} ...")
print()

# [1] DataFrrame 형태로 전달
try:
    loaded.predict(reversed_X)
    print(" * DataFrame 통과 * ")

except Exception as e:
    print(f" * DataFrame : {type(e).__name__}")
    print(str(e))

# [2] numpy 배열로 전달
with warnings.catch_warnings():
    warnings.simplefilter("ignore")     # 경고만 끄고 실행
    pred_org = loaded.predict(X_test.values)
    pred_reversed = loaded.predict(reversed_X.values)

print(" * numpy 배열 *")
print(f" original : {len(pred_org):,} / reversed : {len(pred_reversed):,}")
print(f" 예측이 달라진 건수 : {(pred_org != pred_reversed).sum():,}")

원래 순서 : ['ret_1d', 'ret_5d', 'ma5_ratio'] ...
뒤집은 순서 : ['range_pct', 'volume_ratio', 'vol20'] ...

 * DataFrame : ValueError
The feature names should match those that were passed during fit.
Feature names must be in the same order as they were in fit.

 * numpy 배열 *
 original : 2,000 / reversed : 2,000
 예측이 달라진 건수 : 1,231


> **DataFrame** 형태로 전달하면, 사이킷런에서 열 이름을 확인하고\
> 순서가 잘못 된 경우 예측을 막아줌 (예외를 발생 시킴)

> **numpy 배열** 형태로 전달하면, 사이킷런에서도 막아주지 못함\
> 배열에서는 열 이름을 확인 할 수 없으므로 순서를 확인 할 방법이 없음

**모델을 저장할 때 피처 목록도 함께 저장해야 함**\
예측 전에 저장 된 순서대로 열을 정렬하여 진행할 수 있음

---

## 모델 저장 시 함께 남겨야 하는 것
모델 파일 하나만 지정하면, 시간이 흐른 후 (몇달 뒤) 기억할 수 있을까?

- **전처리기 (Pipeline)**
    - 순서가 바뀌면 엉뚱한 예측을 할 수 있음

- **피처 목록과 순서**
    - 순서가 바뀌면 엉뚱한 예측을 할 수 있음

- 학습 데이터 기간
    - 언제까지의 데이터로 학습을 시켰는지

- 성능 지표
    - 나중에 새로 학습한 모델과의 비교 기준

- 라이브러리 버전
    - 버전이 다를 경우 로드 실패 될 수 있음

> 저장 할 때 위의 정보들을 함께 pkl 형태로 저장할 수 있으나,\
> 메타 데이터는 `meta.json` 처럼 사람이 해석할 수 있는 파일로 따로 저장해 두는 것이 좋음

In [20]:
import sklearn

bundle = {
    "pipeline": pipe,                                                   # 전처리 + 모델
    "features": FEATURES,                                               # 피처 목록과 순서
    "train_period": (tr["date"].min().date(), tr["date"].max().date()), # 학습 기간
    "test_accuracy": pipe.score(X_test, y_test),                        # 성능 지표
    "sklearn_version": sklearn.__version__,                             # 라이브러리 버전
    "create_at": time.strftime("%Y-%m-%d %H:%M:%S"),                    # 저장 날짜
}

joblib.dump(bundle, "models/model_bundle.pkl")
print("저장 : models/model_bundle.pkl"
      f" ({os.path.getsize('models/model_bundle.pkl') / 1024 :,.0f} KB)")

저장 : models/model_bundle.pkl (1,506 KB)


In [23]:
# -- 수료 후 ------------------------------------------------------------
b = joblib.load("models/model_bundle.pkl")

print("이 파일에 저장 된 정보.. (모델 정보)")
print(f" 학습 기간 : {b['train_period'][0]} ~ {b['train_period'][1]}")
print(f" 테스트 정확도 : {b['test_accuracy']:.4f}")
print(f" 라이브러리 버전 : {b['sklearn_version']} (현재: {sklearn.__version__})")
print(f" 저장 날짜 : {b['create_at']}")
print(f" 피처 : {len(b['features'])} : {b['features'][:3]} ...")

pred = b['pipeline'].predict(X_test.head(5))
print(f"예측 결과 : ... {pred}")

이 파일에 저장 된 정보.. (모델 정보)
 학습 기간 : 2023-10-24 ~ 2026-01-16
 테스트 정확도 : 0.5005
 라이브러리 버전 : 1.9.1 (현재: 1.9.1)
 저장 날짜 : 2026-10-15 10:42:26
 피처 : 7 : ['ret_1d', 'ret_5d', 'ma5_ratio'] ...
예측 결과 : ... [1 0 0 0 0]
